# 06 — Temporal Difference Learning: TD(0), SARSA, and Q-learning

Notebook 03 evaluated a policy by running episodes to the end and averaging the
returns. It was honest about what that cost:

> $1/\sqrt{N}$ convergence — slow next to dynamic programming
> each state's estimate is independent of the others' errors, since nothing bootstraps

**Bootstrapping** is the missing idea, and this notebook supplies it. Instead of
waiting for an episode to finish, we update a state's value from the very next
reward plus our *current guess* about where we landed — learning from a guess,
and improving the guess as we go.

That single change buys three things:

1. **Learning without waiting.** Updates happen every step, not every episode.
2. **Control without a model.** Learning $q(s, a)$ rather than $v(s)$ means the
   greedy action can be read straight off the table — no $P$ required. Notebook
   02 needed the model to extract a policy; this one does not.
3. **A choice about what you are learning.** SARSA and Q-learning differ by one
   symbol and converge to genuinely different answers. Most of this notebook is
   about that difference, because it is the one that gets misunderstood.

Everything is still graded against notebook 02's exact answers.

In [1]:
# --- environment from notebook 01, repeated so this notebook stands alone ---
from __future__ import annotations

from enum import IntEnum
from typing import NamedTuple

import numpy as np

class State(IntEnum):
    NO_INFO = 0           # nothing checked yet
    SALINITY_CHECKED = 1  # feed salinity known
    FOULING_CHECKED = 2   # fouling indicators known
    BOTH_CHECKED = 3      # both kinds of evidence in hand
    SUCCESS = 4           # problem solved (terminal)
    FAILURE = 5           # wrong or unsafe fix submitted (terminal)


class Action(IntEnum):
    CHECK_SALINITY = 0
    CHECK_FOULING = 1
    RUN_SIMULATION = 2
    SUBMIT_DIRECTLY = 3


N_STATES, N_ACTIONS = len(State), len(Action)
TERMINAL_STATES = frozenset({State.SUCCESS, State.FAILURE})
NONTERMINAL = [s for s in State if s not in TERMINAL_STATES]


def is_terminal(state) -> bool:
    return State(state) in TERMINAL_STATES


COST_CHECK = -0.5          # first look at a piece of evidence
COST_REPEAT_CHECK = -1.0   # re-checking something already known: pure waste
REWARD_SIM_SUCCESS = 9.0   # +10 outcome, minus the -1 implicit cost of simulating
REWARD_SIM_FAILURE = -11.0
REWARD_SUBMIT_SUCCESS = 10.0
REWARD_SUBMIT_FAILURE = -10.0

SIM_SUCCESS_PROB = {
    State.NO_INFO: 0.15,
    State.SALINITY_CHECKED: 0.55,
    State.FOULING_CHECKED: 0.45,
    State.BOTH_CHECKED: 0.95,
}
SUBMIT_SUCCESS_PROB = {
    State.NO_INFO: 0.05,
    State.SALINITY_CHECKED: 0.35,
    State.FOULING_CHECKED: 0.25,
    State.BOTH_CHECKED: 0.75,
}


class Transition(NamedTuple):
    prob: float
    next_state: State
    reward: float


# Evidence held in each non-terminal state, used to work out where a check lands.
_EVIDENCE = {
    State.NO_INFO: frozenset(),
    State.SALINITY_CHECKED: frozenset({Action.CHECK_SALINITY}),
    State.FOULING_CHECKED: frozenset({Action.CHECK_FOULING}),
    State.BOTH_CHECKED: frozenset({Action.CHECK_SALINITY, Action.CHECK_FOULING}),
}
_STATE_BY_EVIDENCE = {ev: st for st, ev in _EVIDENCE.items()}


def transitions(state, action) -> tuple[Transition, ...]:
    # Every outcome of taking `action` in `state`, probabilities summing to 1.
    state, action = State(state), Action(action)

    if is_terminal(state):
        return (Transition(1.0, state, 0.0),)

    if action in (Action.CHECK_SALINITY, Action.CHECK_FOULING):
        already_known = action in _EVIDENCE[state]
        next_state = (
            state if already_known
            else _STATE_BY_EVIDENCE[_EVIDENCE[state] | {action}]
        )
        reward = COST_REPEAT_CHECK if already_known else COST_CHECK
        return (Transition(1.0, next_state, reward),)

    if action is Action.RUN_SIMULATION:
        p = SIM_SUCCESS_PROB[state]
        return (
            Transition(p, State.SUCCESS, REWARD_SIM_SUCCESS),
            Transition(1.0 - p, State.FAILURE, REWARD_SIM_FAILURE),
        )

    p = SUBMIT_SUCCESS_PROB[state]
    return (
        Transition(p, State.SUCCESS, REWARD_SUBMIT_SUCCESS),
        Transition(1.0 - p, State.FAILURE, REWARD_SUBMIT_FAILURE),
    )


def transition_tables() -> tuple[np.ndarray, np.ndarray]:
    # Dense tables for exact methods: P[s, a, s'] and expected R[s, a].
    P = np.zeros((N_STATES, N_ACTIONS, N_STATES))
    R = np.zeros((N_STATES, N_ACTIONS))
    for s in State:
        for a in Action:
            for prob, next_state, reward in transitions(s, a):
                P[s, a, next_state] += prob
                R[s, a] += prob * reward
    return P, R


P, R = transition_tables()


def step(state, action, rng) -> tuple[State, float, bool]:
    # Sample one environment step: (next_state, reward, done).
    outcomes = transitions(state, action)
    if len(outcomes) == 1:
        only = outcomes[0]
        return only.next_state, only.reward, is_terminal(only.next_state)
    u, cumulative = rng.random(), 0.0
    for prob, next_state, reward in outcomes:
        cumulative += prob
        if u < cumulative:
            return next_state, reward, is_terminal(next_state)
    last = outcomes[-1]  # float-rounding fallback
    return last.next_state, last.reward, is_terminal(last.next_state)


GAMMA = 0.95

print("setup complete;", N_STATES, "states,", N_ACTIONS, "actions, gamma =", GAMMA)

setup complete; 6 states, 4 actions, gamma = 0.95


## The answer key, recomputed

Notebook 02 solved this MDP exactly. We need two things from it: $v^*$ to grade
prediction against, and $q^*$ — the **action** values — to grade control against.
Value iteration gives both, and the last sweep of it is exactly the $q$ table.

In [2]:
def value_iteration(gamma=GAMMA, tol=1e-14, max_sweeps=10_000):
    """Iterate the Bellman optimality operator to its fixed point."""
    V = np.zeros(N_STATES)
    for _ in range(max_sweeps):
        Q = R + gamma * P @ V
        V_next = Q.max(axis=1)
        V_next[list(TERMINAL_STATES)] = 0.0
        if np.abs(V_next - V).max() < tol:
            return V_next, R + gamma * P @ V_next
        V = V_next
    raise RuntimeError("value iteration did not converge")


V_STAR, Q_STAR = value_iteration()

OPTIMAL_ACTIONS = {
    State.NO_INFO: {Action.CHECK_SALINITY, Action.CHECK_FOULING},
    State.SALINITY_CHECKED: {Action.CHECK_FOULING},
    State.FOULING_CHECKED: {Action.CHECK_SALINITY},
    State.BOTH_CHECKED: {Action.RUN_SIMULATION},
}

# The same numbers notebook 02 printed.
assert np.allclose(V_STAR, [6.245, 7.1, 7.1, 8.0, 0.0, 0.0])

print("q*  (exact action values, gamma = 0.95)\n")
print(f"{'state':<19}" + "".join(f"{a.name:>17}" for a in Action))
for s in NONTERMINAL:
    row = "".join(f"{Q_STAR[s, a]:>17.3f}" for a in Action)
    print(f"{State(s).name:<19}{row}")
print("\nv* = max_a q*(s, a) =", np.round(V_STAR[:4], 3))

q*  (exact action values, gamma = 0.95)

state                 CHECK_SALINITY    CHECK_FOULING   RUN_SIMULATION  SUBMIT_DIRECTLY
NO_INFO                        6.245            6.245           -8.000           -9.000
SALINITY_CHECKED               5.745            7.100            0.000           -3.000
FOULING_CHECKED                7.100            5.745           -2.000           -5.000
BOTH_CHECKED                   6.600            6.600            8.000            5.000

v* = max_a q*(s, a) = [6.245 7.1   7.1   8.   ]


Read the top row: from `NO_INFO`, either check is worth $6.245$ and both terminal
actions are catastrophic ($-8.0$ for simulating blind, $-9.0$ for submitting
blind). The two checks are **exactly tied** — the evidence commutes, so $q^*$ has
no opinion about which to do first. Hold onto that tie; it comes back.

## Learning from a guess

Monte Carlo waits for the episode to end and uses the actual return:

$$V(S_t) \leftarrow V(S_t) + \alpha\,[\,\underbrace{G_t}_{\text{actual return}} - V(S_t)\,]$$

TD(0) does not wait. It takes one step, observes one reward, and substitutes its
own current estimate for everything after that:

$$V(S_t) \leftarrow V(S_t) + \alpha\,[\,\underbrace{R_{t+1} + \gamma V(S_{t+1})}_{\text{TD target}} - V(S_t)\,]$$

The bracketed quantity is the **TD error** $\delta_t$. The substitution is
justified by the Bellman expectation equation from notebook 02 —
$v_\pi(s) = \mathbb{E}[R_{t+1} + \gamma v_\pi(S_{t+1})]$ — with the crucial
caveat that we are plugging in $V$, not $v_\pi$. Early on $V$ is wrong, so the
target is wrong: **TD is biased**. What it buys is variance. $G_t$ carries every
coin flip between here and termination; the TD target carries exactly one.

Bias against variance, one step at a time. Here are both targets for a single
episode.

In [3]:
def careful(state):
    """The optimal policy from notebook 02: gather both, then validate."""
    return {
        State.NO_INFO: Action.CHECK_SALINITY,
        State.SALINITY_CHECKED: Action.CHECK_FOULING,
        State.FOULING_CHECKED: Action.CHECK_SALINITY,
        State.BOTH_CHECKED: Action.RUN_SIMULATION,
    }[State(state)]


def episode(policy_fn, rng, max_steps=20):
    s, traj = State.NO_INFO, []
    for _ in range(max_steps):
        a = policy_fn(s)
        ns, r, done = step(s, a, rng)
        traj.append((s, a, r, ns, done))
        s = ns
        if done:
            break
    return traj


rng = np.random.default_rng(4)
traj = episode(careful, rng)

# Monte Carlo targets: discounted return from each step to the end.
G, returns = 0.0, []
for _, _, r, _, _ in reversed(traj):
    G = r + GAMMA * G
    returns.append(G)
returns.reverse()

# TD targets, using a V that already knows the answer, to isolate the idea.
V_oracle = V_STAR.copy()

print(f"{'step':<6}{'state':<19}{'reward':>8}{'MC target G_t':>15}{'TD target':>12}")
for i, (s, a, r, ns, done) in enumerate(traj):
    td_target = r + GAMMA * (0.0 if done else V_oracle[ns])
    print(f"{i:<6}{State(s).name:<19}{r:>8.1f}{returns[i]:>15.3f}{td_target:>12.3f}")
print(f"\nthis episode's outcome: {State(traj[-1][3]).name}")
print("MC targets swing with the terminal coin flip; TD targets do not.")

step  state                reward  MC target G_t   TD target
0     NO_INFO                -0.5          7.147       6.245
1     SALINITY_CHECKED       -0.5          8.050       7.100
2     BOTH_CHECKED            9.0          9.000       9.000

this episode's outcome: SUCCESS
MC targets swing with the terminal coin flip; TD targets do not.


Run that cell again after changing the seed. **Every** row of the MC column
moves, because every state's target inherits the $\pm 20$ swing of the terminal
outcome. Only the *last* row of the TD column moves — the earlier transitions are
deterministic, so their targets are fixed no matter how the episode ends. That
containment is the variance reduction, made concrete: noise enters a TD target
from one step away, not from the whole remaining future.

Of course, the TD column above cheated: it used $v^*$ as the guess. Real TD(0)
starts from zeros and has to bootstrap its way up.

## TD(0) and Monte Carlo, implemented

Both estimate $v_\pi$ for the same fixed policy, both with a constant step size,
so the comparison is like for like.

In [4]:
def mc_predict(n_episodes, rng, policy_fn=careful, alpha=0.05):
    """Every-visit Monte Carlo with a constant step size."""
    V = np.zeros(N_STATES)
    for _ in range(n_episodes):
        traj = episode(policy_fn, rng)
        G = 0.0
        for s, _, r, _, _ in reversed(traj):
            G = r + GAMMA * G
            V[s] += alpha * (G - V[s])
    return V


def td0_predict(n_episodes, rng, policy_fn=careful, alpha=0.05):
    """TD(0): update every step from the reward plus the current guess."""
    V = np.zeros(N_STATES)
    for _ in range(n_episodes):
        s = State.NO_INFO
        while not is_terminal(s):
            a = policy_fn(s)
            ns, r, done = step(s, a, rng)
            target = r + GAMMA * (0.0 if done else V[ns])
            V[s] += alpha * (target - V[s])
            s = ns
    return V


# The careful policy always checks salinity first, so FOULING_CHECKED is never
# reached. Grading it would just measure a state neither method ever sees --
# exactly the "no samples" problem notebook 03 ran into.
VISITED = [State.NO_INFO, State.SALINITY_CHECKED, State.BOTH_CHECKED]


def rmse(V, states=VISITED):
    idx = [int(s) for s in states]
    return float(np.sqrt(np.mean((V[idx] - V_STAR[idx]) ** 2)))


print("300 episodes, alpha = 0.05, seed 0\n")
print(f"{'state':<19}{'v_pi (exact)':>14}{'MC':>10}{'TD(0)':>10}")
V_mc = mc_predict(300, np.random.default_rng(0))
V_td = td0_predict(300, np.random.default_rng(0))
for s in VISITED:
    print(f"{State(s).name:<19}{V_STAR[s]:>14.3f}{V_mc[s]:>10.3f}{V_td[s]:>10.3f}")
print(f"\n{'RMSE':<19}{'':>14}{rmse(V_mc):>10.4f}{rmse(V_td):>10.4f}")

300 episodes, alpha = 0.05, seed 0

state                v_pi (exact)        MC     TD(0)
NO_INFO                     6.245     6.063     5.610
SALINITY_CHECKED            7.100     6.909     6.363
BOTH_CHECKED                8.000     7.799     7.799

RMSE                                 0.1916    0.5737


## Head to head

The single run above put Monte Carlo three times ahead at 300 episodes. Before
concluding anything from that: one seed proves nothing, a point notebook 03 made
at length and then applied to its own experiments. Fifty seeds per budget, then —
and note that the aggregate reverses the verdict that single run suggested.

In [5]:
BUDGETS = [5, 10, 30, 100, 300, 1000]
SEEDS = 50

print(f"RMSE over the three visited states, mean of {SEEDS} seeds, alpha = 0.05\n")
print(f"{'episodes':>9}{'MC':>10}{'TD(0)':>10}{'MC / TD':>10}   winner")
for n in BUDGETS:
    mc = np.mean([rmse(mc_predict(n, np.random.default_rng(s))) for s in range(SEEDS)])
    td = np.mean([rmse(td0_predict(n, np.random.default_rng(s))) for s in range(SEEDS)])
    winner = "TD" if td < mc else "MC"
    print(f"{n:>9}{mc:>10.4f}{td:>10.4f}{mc / td:>10.2f}   {winner}")

RMSE over the three visited states, mean of 50 seeds, alpha = 0.05

 episodes        MC     TD(0)   MC / TD   winner
        5    5.4742    6.5209      0.84   MC
       10    4.3143    6.0127      0.72   MC
       30    1.6829    4.1444      0.41   MC
      100    0.5625    0.6761      0.83   MC


      300    0.5968    0.5080      1.17   TD


     1000    0.5931    0.5144      1.15   TD


**TD loses the early rounds and wins the late ones, and the margin is small
either way.** At 30 episodes Monte Carlo is more than twice as accurate. The
crossover is somewhere near 100–300 episodes, after which TD leads by roughly
15%.

This is worth sitting with, because the textbook framing is "TD converges faster
than MC" and that is not what happened for the first hundred episodes.

Two things are fighting:

- **TD's variance advantage** is real but *small here*, because episodes are
  three steps long. The return $G_t$ only accumulates a handful of random
  events, so it is not the wildly noisy target it becomes in a long episode.
- **TD's bias is expensive at the start.** $V$ begins at zeros, so early TD
  targets are $R_{t+1} + \gamma \cdot 0$ — pure immediate reward. Value has to
  propagate backwards from the terminal states one link per update, and until it
  arrives, states far from the end are being pulled toward numbers that are
  simply too small.

The second effect is visible directly.

In [6]:
print("mean estimate after 100 episodes, 50 seeds\n")
mc = np.mean([mc_predict(100, np.random.default_rng(s)) for s in range(SEEDS)], axis=0)
td = np.mean([td0_predict(100, np.random.default_rng(s)) for s in range(SEEDS)], axis=0)
print(f"{'state':<19}{'steps to end':>13}{'v_pi':>9}{'MC':>9}{'TD(0)':>9}{'TD error':>10}")
for s, dist in zip(VISITED, [3, 2, 1]):
    print(f"{State(s).name:<19}{dist:>13}{V_STAR[s]:>9.3f}"
          f"{mc[s]:>9.3f}{td[s]:>9.3f}{td[s] - V_STAR[s]:>+10.3f}")

mean estimate after 100 episodes, 50 seeds

state               steps to end     v_pi       MC    TD(0)  TD error
NO_INFO                        3    6.245    6.221    5.440    -0.805
SALINITY_CHECKED               2    7.100    7.072    6.882    -0.218
BOTH_CHECKED                   1    8.000    7.967    7.967    -0.033


The TD error grows with distance from the terminal state, and it is **negative**
at every one of them: TD is systematically *under*-estimating, exactly as the
propagation story predicts. `BOTH_CHECKED` is one action from the end and TD has
it almost exactly right; `NO_INFO` is two actions further back and is still well
short. Monte Carlo shows **no such gradient** — its error sits near $-0.03$
everywhere regardless of distance, which is just the residue of starting at zero
with a constant step size. Lag is TD's alone, and it is proportional to how far
the value has to travel.

So the honest summary for *this* MDP: TD's advantage is real but modest, and it
has to pay off a start-up debt first. The advantage grows with episode length,
which is precisely the regime this tiny problem does not have.

## From prediction to control

Everything so far estimates $v_\pi$ for a policy someone handed us. To *improve*
a policy we need to compare actions, and $v$ cannot do that without a model:
picking the best action from $v$ means evaluating
$\sum_{s'} P(s' \mid s, a)[R + \gamma v(s')]$, and $P$ is exactly what we do not
have.

Learning $q(s, a)$ instead removes the problem. The greedy action is
$\arg\max_a q(s, a)$ — a table lookup, no model anywhere. This is the real reason
action values dominate model-free RL.

But estimating $q$ means every action has to be *tried*, including the bad ones,
and a greedy policy tries nothing new. The standard compromise is
$\varepsilon$-greedy: act greedily with probability $1 - \varepsilon$, act at
random otherwise.

In [7]:
def eps_greedy(Q, s, eps, rng):
    if rng.random() < eps:
        return Action(int(rng.integers(N_ACTIONS)))
    return Action(int(np.argmax(Q[s])))


def visit_counts(n_episodes, rng, eps=0.1, alpha=0.1, exploring_starts=False):
    """Q-learning, but reporting how often each state was actually seen."""
    Q = np.zeros((N_STATES, N_ACTIONS))
    counts = np.zeros(N_STATES)
    starts = [int(s) for s in NONTERMINAL]
    for _ in range(n_episodes):
        s = State(int(rng.choice(starts))) if exploring_starts else State.NO_INFO
        while True:
            counts[s] += 1
            a = eps_greedy(Q, s, eps, rng)
            ns, r, done = step(s, a, rng)
            Q[s, a] += alpha * ((r if done else r + GAMMA * Q[ns].max()) - Q[s, a])
            if done:
                break
            s = ns
    return counts


print("state visits over 2,000 episodes of eps-greedy Q-learning (eps = 0.1)\n")
print(f"{'starts':<22}" + "".join(f"{State(s).name:>19}" for s in NONTERMINAL))
for es in (False, True):
    c = visit_counts(2000, np.random.default_rng(0), exploring_starts=es)
    label = "exploring starts" if es else "always NO_INFO"
    print(f"{label:<22}" + "".join(f"{int(c[s]):>19}" for s in NONTERMINAL))

state visits over 2,000 episodes of eps-greedy Q-learning (eps = 0.1)

starts                            NO_INFO   SALINITY_CHECKED    FOULING_CHECKED       BOTH_CHECKED
always NO_INFO                       2000               1671                304               1981
exploring starts                      509                974                529               2090


`FOULING_CHECKED` is visited a few hundred times against two thousand for the
start state — it is only reached when $\varepsilon$ fires *and* picks
`CHECK_FOULING` first. Its $q$ row is therefore learned from a fraction of the
data, and any conclusion drawn about it is correspondingly weak.

This is the same lesson notebook 03 met when a deterministic policy left
`FOULING_CHECKED` with no samples at all and the estimate came back `NaN`.
$\varepsilon$-greedy upgrades "never" to "rarely", which is better but not
solved. **Exploring starts** — beginning episodes from a random non-terminal
state — flattens the visit distribution properly, and everything below uses them.

## SARSA and Q-learning

The two algorithms differ in one term. Having taken $A_t$ in $S_t$ and landed in
$S_{t+1}$:

$$\text{SARSA:}\qquad Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha\,[\,R_{t+1} + \gamma\, Q(S_{t+1}, \mathbf{A_{t+1}}) - Q(S_t, A_t)\,]$$

$$\text{Q-learning:}\qquad Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha\,[\,R_{t+1} + \gamma\, \max_{a} Q(S_{t+1}, a) - Q(S_t, A_t)\,]$$

SARSA bootstraps from the action it **actually took next** — including the
random ones. Q-learning bootstraps from the action it **would have taken if it
were greedy**. SARSA is therefore *on-policy*: it evaluates the
$\varepsilon$-greedy policy it is following. Q-learning is *off-policy*: it
evaluates the greedy policy while following a different one.

In [8]:
def learn(algo, n_episodes, seed, alpha=0.1, eps=0.1, exploring_starts=True):
    """SARSA or Q-learning. The two differ only in the bootstrap term."""
    rng = np.random.default_rng(seed)
    Q = np.zeros((N_STATES, N_ACTIONS))
    starts = [int(s) for s in NONTERMINAL]
    for _ in range(n_episodes):
        s = State(int(rng.choice(starts))) if exploring_starts else State.NO_INFO
        a = eps_greedy(Q, s, eps, rng)
        while True:
            ns, r, done = step(s, a, rng)
            if done:
                Q[s, a] += alpha * (r - Q[s, a])
                break
            if algo == "sarsa":
                na = eps_greedy(Q, ns, eps, rng)          # the action taken
                Q[s, a] += alpha * (r + GAMMA * Q[ns, na] - Q[s, a])
                s, a = ns, na
            else:
                Q[s, a] += alpha * (r + GAMMA * Q[ns].max() - Q[s, a])  # the best action
                s = ns
                a = eps_greedy(Q, s, eps, rng)
    return Q


Q_sarsa = learn("sarsa", 3000, seed=0)
Q_qlearn = learn("qlearn", 3000, seed=0)

def greedy_actions(Q):
    return {State(s): Action(int(Q[s].argmax())) for s in NONTERMINAL}

for name, Q in [("SARSA", Q_sarsa), ("Q-learning", Q_qlearn)]:
    picks = greedy_actions(Q)
    ok = sum(picks[s] in OPTIMAL_ACTIONS[s] for s in NONTERMINAL)
    print(f"{name:<12} greedy policy after 3,000 episodes  ({ok}/4 optimal)")
    for s in NONTERMINAL:
        mark = "ok" if picks[s] in OPTIMAL_ACTIONS[s] else "WRONG"
        print(f"    {State(s).name:<19}{picks[s].name:<17}{mark}")
    print()

SARSA        greedy policy after 3,000 episodes  (4/4 optimal)
    NO_INFO            CHECK_FOULING    ok
    SALINITY_CHECKED   CHECK_FOULING    ok
    FOULING_CHECKED    CHECK_SALINITY   ok
    BOTH_CHECKED       RUN_SIMULATION   ok

Q-learning   greedy policy after 3,000 episodes  (4/4 optimal)
    NO_INFO            CHECK_SALINITY   ok
    SALINITY_CHECKED   CHECK_FOULING    ok
    FOULING_CHECKED    CHECK_SALINITY   ok
    BOTH_CHECKED       RUN_SIMULATION   ok



Both find the optimal policy. That is the *uninteresting* half of the answer, and
it is where most treatments stop. The interesting half is that the two tables
underneath those identical policies are not converging to the same numbers.

## What each one is actually converging to

Q-learning converges to $q^*$. SARSA converges to $q^\pi$ where $\pi$ is the
$\varepsilon$-greedy policy it is following — which is *not* optimal, because it
takes a random action one time in $\varepsilon$.

We can compute that second target exactly, the same way notebook 02 computed
$q^*$: iterate a Bellman operator to its fixed point, but back up through the
$\varepsilon$-greedy action distribution instead of a $\max$.

$$q_\varepsilon(s, a) = R(s,a) + \gamma \sum_{s'} P(s' \mid s, a) \sum_{a'} \pi_\varepsilon(a' \mid s')\, q_\varepsilon(s', a')$$

With an exact target for each algorithm, the claim stops being folklore and
becomes a measurement.

In [9]:
def eps_greedy_fixed_point(eps, gamma=GAMMA, tol=1e-14, max_sweeps=100_000):
    """q for the policy that is eps-greedy with respect to itself."""
    Q = np.zeros((N_STATES, N_ACTIONS))
    for _ in range(max_sweeps):
        pi = np.full((N_STATES, N_ACTIONS), eps / N_ACTIONS)
        pi[np.arange(N_STATES), Q.argmax(axis=1)] += 1 - eps
        V = np.einsum("sa,sa->s", pi, Q)
        V[list(TERMINAL_STATES)] = 0.0
        Q_next = R + gamma * P @ V
        if np.abs(Q_next - Q).max() < tol:
            return Q_next
        Q = Q_next
    raise RuntimeError("did not converge")


EPS, ALPHA, N_EP, N_SEEDS = 0.2, 0.02, 20_000, 6
Q_EPS = eps_greedy_fixed_point(EPS)

Q_s = np.mean([learn("sarsa", N_EP, s, ALPHA, EPS) for s in range(N_SEEDS)], axis=0)
Q_q = np.mean([learn("qlearn", N_EP, s, ALPHA, EPS) for s in range(N_SEEDS)], axis=0)

print(f"eps = {EPS}, alpha = {ALPHA}, {N_EP:,} episodes, mean of {N_SEEDS} seeds\n")
print(f"{'state':<19}{'action':<17}{'q*':>8}{'q_eps':>8}{'SARSA':>9}{'Q-learn':>9}")
for s in NONTERMINAL:
    for a in Action:
        print(f"{State(s).name:<19}{Action(a).name:<17}"
              f"{Q_STAR[s, a]:>8.3f}{Q_EPS[s, a]:>8.3f}{Q_s[s, a]:>9.3f}{Q_q[s, a]:>9.3f}")

idx = np.ix_([int(s) for s in NONTERMINAL], range(N_ACTIONS))
print(f"\n{'':<12}{'vs q*':>10}{'vs q_eps':>11}   (max abs error)")
print(f"{'SARSA':<12}{np.abs(Q_s - Q_STAR)[idx].max():>10.3f}"
      f"{np.abs(Q_s - Q_EPS)[idx].max():>11.3f}")
print(f"{'Q-learning':<12}{np.abs(Q_q - Q_STAR)[idx].max():>10.3f}"
      f"{np.abs(Q_q - Q_EPS)[idx].max():>11.3f}")

eps = 0.2, alpha = 0.02, 20,000 episodes, mean of 6 seeds

state              action                 q*   q_eps    SARSA  Q-learn
NO_INFO            CHECK_SALINITY      6.245   5.062    5.294    6.210
NO_INFO            CHECK_FOULING       6.245   4.862    4.517    6.217
NO_INFO            RUN_SIMULATION     -8.000  -8.000   -8.289   -8.245
NO_INFO            SUBMIT_DIRECTLY    -9.000  -9.000   -8.904   -8.924
SALINITY_CHECKED   CHECK_SALINITY      5.745   4.562    4.586    5.745
SALINITY_CHECKED   CHECK_FOULING       7.100   6.796    6.892    7.192
SALINITY_CHECKED   RUN_SIMULATION      0.000   0.000    0.066    0.096
SALINITY_CHECKED   SUBMIT_DIRECTLY    -3.000  -3.000   -3.148   -3.230
FOULING_CHECKED    CHECK_SALINITY      7.100   6.796    6.855    7.132
FOULING_CHECKED    CHECK_FOULING       5.745   4.362    4.064    5.757
FOULING_CHECKED    RUN_SIMULATION     -2.000  -2.000   -1.795   -1.649
FOULING_CHECKED    SUBMIT_DIRECTLY    -5.000  -5.000   -5.102   -5.096
BOTH_CHECKED      

**Each algorithm is close to its own target and far from the other one.** SARSA
sits near $q_\varepsilon$ and is badly wrong about $q^*$; Q-learning does the
reverse, despite the two having followed *the same behaviour policy* and seen
statistically identical data. Off-policy learning is not a detail of
implementation — it changes what the numbers mean.

Look at where $q^*$ and $q_\varepsilon$ agree and where they part:

- `RUN_SIMULATION` and `SUBMIT_DIRECTLY` are **identical** in both columns from
  every state. Those actions terminate the episode immediately, so there is no
  future left in which exploration could go wrong. No future, no discount for it.
- Both checks are **lower** under $q_\varepsilon$. Checking commits you to at
  least one more decision, and an $\varepsilon$-greedy agent will occasionally
  botch it — so the value of continuing is discounted by the cost of your own
  future clumsiness.

That is the on-policy/off-policy distinction in one sentence: **SARSA prices in
the mistakes you are going to make; Q-learning prices in the mistakes you would
avoid if you stopped exploring.**

## Exploration breaks the tie

There is a consequence hiding in that table, and it is the nicest thing this MDP
has to say.

In [10]:
s = State.NO_INFO
print("from NO_INFO:\n")
print(f"{'':<18}{'q*':>9}{'q_eps':>9}")
for a in (Action.CHECK_SALINITY, Action.CHECK_FOULING):
    print(f"{a.name:<18}{Q_STAR[s, a]:>9.3f}{Q_EPS[s, a]:>9.3f}")

gap_star = abs(Q_STAR[s, Action.CHECK_SALINITY] - Q_STAR[s, Action.CHECK_FOULING])
gap_eps = abs(Q_EPS[s, Action.CHECK_SALINITY] - Q_EPS[s, Action.CHECK_FOULING])
print(f"\n  gap under q*     = {gap_star:.6f}   (exactly tied)")
print(f"  gap under q_eps  = {gap_eps:.6f}   (salinity strictly better)")
print(f"\n  P(simulation succeeds) with salinity only = {SIM_SUCCESS_PROB[State.SALINITY_CHECKED]}")
print(f"  P(simulation succeeds) with fouling  only = {SIM_SUCCESS_PROB[State.FOULING_CHECKED]}")

from NO_INFO:

                         q*    q_eps
CHECK_SALINITY        6.245    5.062
CHECK_FOULING         6.245    4.862

  gap under q*     = 0.000000   (exactly tied)
  gap under q_eps  = 0.199475   (salinity strictly better)

  P(simulation succeeds) with salinity only = 0.55
  P(simulation succeeds) with fouling  only = 0.45


$q^*$ is **exactly indifferent** between the two checks — notebook 02 pointed out
that the optimal policy is not unique for precisely this reason. $q_\varepsilon$
is not indifferent at all. Checking salinity first is strictly better.

The reason is that an optimal agent always makes it to `BOTH_CHECKED`, where the
order it arrived in is forgotten. An $\varepsilon$-greedy agent might be derailed
into acting early — and if it is, it would rather be holding salinity (55%
simulation success) than fouling (45%). **Order only matters if you might not
finish.**

Nothing about the environment changed. The tie was always an artifact of assuming
perfect execution, and letting the agent be fallible is what exposed it. Every
$q^*$ tie is like this: a place where the model is telling you the choice is
free, on the assumption that nothing later goes wrong.

## Does the distinction change the policy?

In [11]:
def policy_value(action_of_state, gamma=GAMMA):
    """Exact value of a deterministic policy: solve (I - gamma P_pi) v = R_pi."""
    pi = np.zeros((N_STATES, N_ACTIONS))
    for s in NONTERMINAL:
        pi[s, action_of_state(s)] = 1.0
    P_pi = np.einsum("sa,sat->st", pi, P)
    R_pi = np.einsum("sa,sa->s", pi, R)
    keep = [int(s) for s in NONTERMINAL]
    v = np.zeros(N_STATES)
    v[keep] = np.linalg.solve(
        np.eye(len(keep)) - gamma * P_pi[np.ix_(keep, keep)], R_pi[keep]
    )
    return v


def score(Q):
    picks = greedy_actions(Q)
    n_opt = sum(picks[s] in OPTIMAL_ACTIONS[s] for s in NONTERMINAL)
    return n_opt, policy_value(lambda s: picks[State(s)])[State.NO_INFO]


print("greedy policy extracted from each algorithm, mean of 30 seeds")
print("(alpha = 0.1, eps = 0.1, exploring starts)\n")
print(f"{'episodes':>9}{'SARSA opt/4':>13}{'SARSA value':>13}"
      f"{'QL opt/4':>11}{'QL value':>11}")
for n in [100, 300, 1000, 3000]:
    rows = [[score(learn(algo, n, s)) for s in range(30)] for algo in ("sarsa", "qlearn")]
    (so, sv), (qo, qv) = [np.mean(r, axis=0) for r in rows]
    print(f"{n:>9}{so:>13.2f}{sv:>13.4f}{qo:>11.2f}{qv:>11.4f}")
print(f"\noptimal value from NO_INFO = {V_STAR[State.NO_INFO]:.4f}")

greedy policy extracted from each algorithm, mean of 30 seeds
(alpha = 0.1, eps = 0.1, exploring starts)

 episodes  SARSA opt/4  SARSA value   QL opt/4   QL value
      100         3.97       6.1548       3.93     6.0645


      300         4.00       6.2450       3.97     6.1548


     1000         3.97       6.1548       3.97     5.4027


     3000         3.97       6.1548       4.00     6.2450

optimal value from NO_INFO = 6.2450


Both are essentially optimal within a few hundred episodes, and neither is
reliably ahead of the other. **In this MDP the on-policy/off-policy distinction
shows up in the values, not in the policy they produce.**

That is a negative result, and it has a clean diagnosis. The distinction changes
the *policy* only where exploring near the optimal path can be punished —
Sutton and Barto's cliff-walking example is built so that a random step off the
optimal route falls into a chasm, and there SARSA learns a deliberately safer
route while Q-learning walks the edge. Here, every dangerous action
(`SUBMIT_DIRECTLY`, or simulating on thin evidence) **ends the episode**. An
exploratory mistake costs you that episode's return and nothing more; it cannot
drop you somewhere worse and leave you there. No cliff, no divergence in policy.

Which is worth stating plainly: if you had only ever tested these two algorithms
on a problem like this one, you would conclude they were interchangeable, and
you would be wrong for reasons the problem cannot show you.

## Where this leaves us

| Notebook | Needs the model? | What it learns |
| --- | --- | --- |
| 02 | **yes** — $P$ and $R$ given | $v^*$, exactly, by planning |
| 03 | no | $v_\pi$, by averaging complete returns |
| 04–05 | no | a policy, by gradient ascent |
| **06 — this one** | no | $q$, by bootstrapping one step at a time |

Every model-free method here treats the environment as a black box that emits
samples. Notebook 03 said it kept $P$ around "for one purpose only: grading", and
04, 05 and 06 have all done the same.

But look at what we threw away. Every episode of Q-learning above observed a
transition — this state, this action, that reward, that next state — used it for
a single update, and discarded it. Those observations are *exactly* the data you
would need to estimate $P$ and $R$ themselves. And notebook 02 already showed
that with $P$ and $R$ in hand the problem can be solved outright, no sampling
required.

So the obvious question, and the subject of notebook 07: **why not learn the
model?** Count the transitions, build $\hat{P}$ and $\hat{R}$, run notebook 02's
value iteration on the estimate, and plan. It should be dramatically more
sample-efficient than anything here — and the notebook will also show what it
costs you when the estimated model is wrong in a way the agent cannot detect.

### Things worth trying

- Set `alpha=0.5` in `td0_predict` and watch the head-to-head table change. How
  much of TD's late-stage win is the algorithm and how much is the step size?
- Run `learn(..., exploring_starts=False)` and check `FOULING_CHECKED`'s row in
  the $q$ table against $q^*$. How wrong is the state you barely visit?
- Raise `EPS` to `0.5` and recompute `Q_EPS`. Which entries move, and which are
  pinned to $q^*$ no matter how bad the exploration gets?
- Make `SUBMIT_DIRECTLY` non-terminal on failure — send it back to `NO_INFO`
  instead — and re-run the last comparison. That builds the cliff this MDP is
  missing, and SARSA and Q-learning should stop agreeing.